# 06 — Fault isolation and recovery

Branch-A populations are independent of branch B. Trap and recovery counts remain separate. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

rows=[]
try:
    batch=resolve_result_batch('e-iso-4', diagnostic_path=os.environ.get('E_ISO_4_DIR'))
except (FileNotFoundError, RuntimeError, ValueError):
    batch=None
containment=[] if batch is None else [value for _,value in passed_json(batch,'containment.json') if value.get('condition') == 'infinite-loop']
if not containment:
    rows.append(pending_record('epoch recovery: infinite-loop','no passed containment.json leaf','trap and recovery events'))
else:
    traps=[int(value.get('traps_total',0)) for value in containment]
    recoveries=[sum(int(node.get('recovery_count',0)) for node in value.get('nodes',[])) for value in containment]
    rows.append({
        'question':'epoch recovery: infinite-loop',
        'status':'READY',
        'condition':'infinite-loop',
        'N_runs':len(containment),
        'median_traps':pd.Series(traps).median(),
        'median_recoveries':pd.Series(recoveries).median(),
        'median_branch_a_throughput_msg_s':None,
        'median_branch_a_p95_ns':None,
        'units':'events',
        'uncertainty':'descriptive only',
        'thesis_evidence':False,
    })
try:
    batch=resolve_result_batch('e-iso-7', diagnostic_path=os.environ.get('E_ISO_7_DIR'))
except (FileNotFoundError, RuntimeError, ValueError):
    batch=None
isolation=[] if batch is None else [value for _,value in passed_json(batch,'branch-isolation.json')]
for condition in ['control','panic-attack','epoch-loop-attack']:
    values=[value for value in isolation if value.get('condition') == condition]
    if not values:
        row=pending_record(f'branch-A: {condition}','no passed branch-isolation.json leaf','messages/second and nanoseconds')
        row.update({'condition':condition,'N_runs':0})
        rows.append(row)
    else:
        branches=[value['branches']['branch_a'] for value in values]
        rows.append({
            'question':f'branch-A: {condition}',
            'status':'READY',
            'condition':condition,
            'N_runs':len(values),
            'median_traps':None,
            'median_recoveries':None,
            'median_branch_a_throughput_msg_s':pd.Series([branch['throughput']['mean_messages_per_second'] for branch in branches]).median(),
            'median_branch_a_p95_ns':pd.Series([branch['latency_ns']['p95'] for branch in branches]).median(),
            'units':'messages/second and nanoseconds',
            'uncertainty':'descriptive only',
            'thesis_evidence':False,
        })
df=pd.DataFrame(rows)
independent_runs=int(df.loc[df.status=='READY','N_runs'].sum())
print(f"{evidence_label(independent_runs, 'events, messages/second, nanoseconds', False)}; independent runs across {len(df[df.status=='READY'])} conditions")
display(df)
